# 03 — Model Evaluation & Backtesting

Comprehensive evaluation of the trained stock predictor:
- Classification report & confusion matrix
- ROC curve + AUC
- Precision-Recall curve
- Backtest: model signals vs buy-and-hold benchmark
- Cumulative returns comparison

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'ml-backend'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

from pipeline.ingest import fetch_stock_data
from pipeline.features import build_feature_matrix

sns.set_theme(style='darkgrid')
TICKER = 'AAPL'
MODEL_PATH = os.path.join('..', 'ml-backend', 'artifacts', 'stock_model.joblib')

## 1. Load Model & Rebuild Test Set

In [ ]:
saved = joblib.load(MODEL_PATH)
model = saved['model']
feature_cols = saved['feature_cols']

raw_df = fetch_stock_data(TICKER, period='2y', use_cache=True)
df, _ = build_feature_matrix(raw_df)
X = df[feature_cols].values
y = df['target'].values

split_idx = int(len(X) * 0.8)
X_test, y_test = X[split_idx:], y[split_idx:]
df_test = df.iloc[split_idx:].reset_index(drop=True)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
print(f'Test samples: {len(y_test)} | Stored metrics: {saved["metrics"]}')

## 2. Classification Report

In [ ]:
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Down (0)', 'Up (1)']))

## 3. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred: Down', 'Pred: Up'],
            yticklabels=['True: Down', 'True: Up'])
ax.set_title(f'{TICKER} Confusion Matrix (Test Set)')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 4. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random baseline')
ax.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'{TICKER} XGBoost — ROC Curve')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 5. Precision-Recall Curve

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(recall, precision, color='tomato', lw=2, label=f'AP = {ap:.3f}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'{TICKER} — Precision-Recall Curve')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Backtest: Model Signals vs Buy-and-Hold

Strategy: go long (1x) when model predicts Up, stay in cash (0x) when Down.  
**No transaction costs — illustrative only.**

In [ ]:
# Actual daily returns in the test window
actual_returns = df_test['Close'].pct_change().fillna(0)

# Strategy returns: take signal from day t (predicted at t-1)
strategy_returns = actual_returns * y_pred  # 1=long, 0=cash

cum_strategy = (1 + strategy_returns).cumprod()
cum_buyhold  = (1 + actual_returns).cumprod()

# Sharpe ratio (annualised)
def sharpe(returns, rf=0.045/252):
    excess = returns - rf
    return (excess.mean() / excess.std()) * np.sqrt(252) if excess.std() > 0 else 0

print(f'Strategy cumulative return: {(cum_strategy.iloc[-1]-1)*100:.1f}%  Sharpe: {sharpe(strategy_returns):.2f}')
print(f'Buy & Hold cumulative return: {(cum_buyhold.iloc[-1]-1)*100:.1f}%  Sharpe: {sharpe(actual_returns):.2f}')

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df_test['Date'], cum_strategy, label='ML Strategy', color='steelblue', linewidth=1.5)
ax.plot(df_test['Date'], cum_buyhold,  label='Buy & Hold',  color='tomato',    linewidth=1.5)
ax.axhline(1, color='gray', linestyle='--', linewidth=0.8)
ax.set_title(f'{TICKER} Backtest: ML Strategy vs Buy-and-Hold (Test Period)')
ax.set_xlabel('Date')
ax.set_ylabel('Portfolio Value (starting at 1.0)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Max Drawdown Analysis

In [ ]:
def max_drawdown(cum_returns):
    peak = cum_returns.cummax()
    drawdown = (cum_returns - peak) / peak
    return drawdown.min()

print(f'Max Drawdown — Strategy:   {max_drawdown(cum_strategy)*100:.1f}%')
print(f'Max Drawdown — Buy & Hold: {max_drawdown(cum_buyhold)*100:.1f}%')

# Plot drawdown
peak_s = cum_strategy.cummax()
dd_s = (cum_strategy - peak_s) / peak_s
peak_b = cum_buyhold.cummax()
dd_b = (cum_buyhold - peak_b) / peak_b

fig, ax = plt.subplots(figsize=(16, 4))
ax.fill_between(df_test['Date'], dd_s * 100, 0, alpha=0.5, color='steelblue', label='Strategy DD')
ax.fill_between(df_test['Date'], dd_b * 100, 0, alpha=0.3, color='tomato',    label='Buy&Hold DD')
ax.set_title('Drawdown Comparison (%)')
ax.set_ylabel('Drawdown (%)')
ax.legend()
plt.tight_layout()
plt.show()